# قسمت دوم — تاریخچه واقعی بورس

این دفترچه به سرور عمومی TSETMC متصل می‌شود. در محیط تهیه بسته به علت خطای دسترسی شبکه اجرا نشده است. بازه کامل تاریخی و کامل‌بودن همه روزهای معاملاتی از قبل تضمین نمی‌شود.

## سلول ۱ — ورود کتابخانه‌ها

In [ ]:
import pydara
import pandas as pd
from pydara.tse import Client, ClientConfig, NetworkConfig
print("نسخه:", pydara.__version__)
print("مسیر نصب:", pydara.__file__)
assert pydara.__version__ == "0.7.3a6", "Install this bundle and restart the kernel."

## سلول ۲ — تعریف تابع گزارش

In [ ]:
def report_history(frame):
    print(f"تعداد ردیف‌ها: {len(frame):,}")
    if not frame.empty:
        print("اولین روز موجود:", frame["date"].min().date())
        print("آخرین روز موجود:", frame["date"].max().date())
    columns = ["date", "jdate", "last_price", "closing_price", "volume", "trade_count"]
    display(frame[columns].tail(10))
    display(frame.dtypes.to_frame("نوع داده"))
    print(frame.attrs["pydara"])

## سلول ۳ — تنظیمات

`None` برای ابتدا و انتها یعنی محدوده ارائه‌شده توسط همین سرویس. برای تاریخ شمسی، رشته و تقویم را با هم تغییر بده.

In [ ]:
SYMBOL = "فولاد"
START = None
END = None
CALENDAR = "gregorian"
CONFIG = ClientConfig(network=NetworkConfig(
    operation_timeout=90,
    read_timeout=30,
    request_budget=30,
))

## سلول ۴ — دریافت و بررسی ذخیره

نوشتن خودکار نباید دریافت موفق را متوقف کند. در این نمونه نتیجه `flush` را جدا بررسی می‌کنیم.

In [ ]:
with Client(config=CONFIG) as client:
    df = client.get_history(
        SYMBOL, start=START, end=END,
        calendar=CALENDAR, include_jdate=True,
    )
    storage_report = client.flush(raise_on_error=False)
    storage_status = client.storage_status()
print("وضعیت ذخیره:", storage_status)
print("ثبت تکمیل شد:", storage_report.complete)

## سلول ۵ — نتیجه

In [ ]:
report_history(df)

## سلول ۶ — یک بازه شمسی

این بار در صورت تازه‌بودن کش، پاسخ قبلی بدون دریافت دوباره استفاده می‌شود.

In [ ]:
with Client(config=CONFIG) as client:
    df_range = client.get_history(
        SYMBOL, start="1403-01-01", end="1404-12-29",
        calendar="jalali", include_jdate=True,
    )
report_history(df_range)

دریافت چند نماد و رابط Async در دفترچه آفلاین آزمایش شده‌اند. در صورت خطا ابزار `tools/check_history.py --live --symbol "فولاد"` را اجرا کن و فایل JSON گزارش را ارسال کن. این خروجی قیمت خام است؛ گزینه تعدیل شرکتی در این قسمت هنوز فعال نیست.